# Sleep Disorder Classification — Sleep Health and Lifestyle Dataset

**Goal:** predict `Sleep Disorder` (None / Insomnia / Sleep Apnea) from demographic, lifestyle, and health measurements, using a filter → model-based → RFECV feature selection pipeline.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.feature_selection import f_classif, mutual_info_classif, chi2, RFECV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## 2. Load & Clean Data

In [ ]:
df = pd.read_csv("Sleep_health_and_lifestyle_dataset.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

`Person ID` is a pure identifier and carries no predictive signal, so it's dropped.

In [ ]:
df2 = df.drop(columns=['Person ID'])

`Sleep Disorder` has missing values where a person simply has no diagnosed disorder — this is filled as an explicit `"none"` class rather than treated as missing data.

In [ ]:
df3 = df2.fillna("none")
df3.info()

## 3. Exploratory Data Analysis

In [ ]:
df3['Sleep Disorder'].value_counts()

In [ ]:
sns.countplot(x=df3['Sleep Disorder'])
plt.title("Sleep disorder distribution")
plt.show()

The target is imbalanced (≈58% none / 21% Sleep Apnea / 21% Insomnia) — this is accounted for later with stratified splits and `class_weight='balanced'`.

In [ ]:
gc = df3['Gender'].value_counts().reset_index()
gc.columns = ['gender', 'count']
sns.barplot(data=gc, x='gender', y='count', hue='gender')
plt.title("Gender distribution")
plt.show()

In [ ]:
sns.barplot(x='Sleep Disorder', y='Age', data=df3)
plt.show()

In [ ]:
sns.histplot(data=df3, x='Age', hue='Sleep Disorder', kde=True, element='step')
plt.show()

`BMI Category` contains `"Normal Weight"` and `"Normal"` as separate labels for the same category — merged into one.

In [ ]:
df3['BMI Category'] = df3['BMI Category'].replace('Normal Weight', 'Normal')

In [ ]:
sns.countplot(x='BMI Category', data=df3)
plt.show()

In [ ]:
df3['BMI Category'].unique()

In [ ]:
sns.histplot(data=df3, x='Stress Level', element='step', kde=True, hue='Sleep Disorder')
plt.show()

In [ ]:
sns.kdeplot(data=df3, x='Heart Rate', hue='Sleep Disorder', fill=True)
plt.show()

In [ ]:
sns.kdeplot(data=df3, x='Daily Steps', hue='Sleep Disorder', fill=True)
plt.show()

In [ ]:
sns.scatterplot(data=df3, x='Sleep Duration', y='Quality of Sleep')
plt.show()

In [ ]:
sns.countplot(data=df3, x='BMI Category', hue='Sleep Disorder')
plt.show()

In [ ]:
corr = df3.corr(numeric_only=True)
sns.heatmap(corr, annot=True)
plt.title("Correlation heatmap (raw numeric features)")
plt.show()

## 4. Feature Engineering & Encoding

`Sleep Duration` and `Daily Steps` are highly correlated with `Quality of Sleep` and `Physical Activity Level` respectively — dropped to reduce redundancy.

In [ ]:
df4 = df3.drop(columns=['Sleep Duration', 'Daily Steps'])

`Blood Pressure` is a single `"systolic/diastolic"` string — split into two numeric columns, since each has distinct physiological meaning.

In [ ]:
df4[['Systolic', 'Diastolic']] = df4['Blood Pressure'].str.split('/', expand=True)
df5 = df4.drop(columns=['Blood Pressure'])

Categorical encoding, matched to each variable's nature: `Gender` is binary, `BMI Category` has a real severity order, `Occupation` has no order.

In [ ]:
le = LabelEncoder()
df5['Gender'] = le.fit_transform(df5['Gender'])

In [ ]:
encoder = OrdinalEncoder(categories=[['Normal', 'Overweight', 'Obese']])
df5[['BMI Category']] = encoder.fit_transform(df5[['BMI Category']])

In [ ]:
df5['Systolic'] = df5['Systolic'].astype(int)
df5['Diastolic'] = df5['Diastolic'].astype(int)

In [ ]:
corr = df5.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f')
plt.title("Correlation heatmap (post-encoding)")
plt.show()

In [ ]:
df6 = pd.get_dummies(df5, columns=['Occupation'], dtype=int)

`Sleep Disorder` is nominal (no order between Insomnia/Sleep Apnea/none), so it's label-encoded rather than ordinally encoded.

In [ ]:
le_target = LabelEncoder()
df6['Sleep Disorder'] = le_target.fit_transform(df6['Sleep Disorder'])
dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))

In [ ]:
df6.head()

## 5. Filter-Based Feature Selection

Numeric features vs. target — ANOVA F-test and mutual information.

In [ ]:
X_numeric = df5.select_dtypes(include='number').drop(columns=['BMI Category', 'Gender'])
y = df6['Sleep Disorder']

f_scores, _ = f_classif(X_numeric, y)
mi_scores = mutual_info_classif(X_numeric, y)

filter_numeric = pd.DataFrame({
    'feature': X_numeric.columns,
    'f_score': f_scores,
    'mi_score': mi_scores
}).sort_values('f_score', ascending=False)

filter_numeric

Categorical features (Gender, BMI Category, one-hot Occupation) vs. target — chi-square.

In [ ]:
occupation_cols = [col for col in df6.columns if col.startswith('Occupation')]
X_categorical = df6[['Gender', 'BMI Category'] + occupation_cols]

chi_scores, p_values = chi2(X_categorical, y)

filter_categorical = pd.DataFrame({
    'feature': X_categorical.columns,
    'chi2_score': chi_scores,
    'p_value': p_values
}).sort_values('chi2_score', ascending=False)

filter_categorical

`Occupation_Software Engineer`, `Occupation_Scientist`, and `Occupation_Manager` show no significant association with the target (p > 0.05) — dropped.

In [ ]:
df6 = df6.drop(columns=['Occupation_Software Engineer', 'Occupation_Scientist', 'Occupation_Manager'])

## 6. Train/Test Split

Stratified split to preserve class proportions in both sets, given the target imbalance.

In [ ]:
X = df6.drop(columns=['Sleep Disorder'])
y = df6['Sleep Disorder']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 7. Model-Based Feature Importance

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)
rf.fit(X_train, y_train)

importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

importances

## 8. Recursive Feature Elimination (RFECV)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

estimator = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rfecv = RFECV(
    estimator=estimator,
    step=1,
    cv=cv,
    scoring='f1_macro',
    min_features_to_select=3
)
rfecv.fit(X_train, y_train)

In [ ]:
ranking = pd.DataFrame({
    'feature': X_train.columns,
    'selected': rfecv.support_,
    'rank': rfecv.ranking_
}).sort_values('rank')

print("Optimal number of features:", rfecv.n_features_)
ranking

In [ ]:
plt.figure()
plt.plot(
    range(rfecv.min_features_to_select, len(rfecv.cv_results_['mean_test_score']) + rfecv.min_features_to_select),
    rfecv.cv_results_['mean_test_score']
)
plt.xlabel("Number of features selected")
plt.ylabel("Mean CV F1-macro score")
plt.title("RFECV performance vs. feature count")
plt.show()

## 9. Finalize Feature Set

Cross-checking filter scores, Random Forest importance, and RFECV selection side by side before locking in the final set.

In [ ]:
combined = importances.merge(
    ranking[['feature', 'selected', 'rank']],
    on='feature',
    how='outer'
).sort_values('importance', ascending=False)

combined

RFECV's selection is used as the final answer, since it's the only method that accounts for feature interactions via cross-validated performance rather than scoring each feature alone.

In [ ]:
final_features = list(X_train.columns[rfecv.support_])
final_features

In [ ]:
X_train_final = X_train[final_features]
X_test_final = X_test[final_features]

final_rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
final_rf.fit(X_train_final, y_train)

y_pred_final = final_rf.predict(X_test_final)
print(classification_report(y_test, y_pred_final))

## 10. Model Evaluation

In [ ]:
cm = confusion_matrix(y_test, y_pred_final)
class_names = le_target.classes_

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix — Final Model")
plt.show()

Cross-validated score, to confirm the held-out test result isn't a lucky single split.

In [ ]:
cv_scores = cross_val_score(
    final_rf,
    X_train_final, y_train,
    cv=cv,
    scoring='f1_macro'
)

print("F1-macro per fold:", cv_scores)
print("Mean F1-macro:", cv_scores.mean())
print("Std F1-macro:", cv_scores.std())

## 11. Conclusion

- Final model: Random Forest on **15 features** selected by RFECV (down from ~20 after one-hot encoding), combining physiological measurements (Systolic, Diastolic, Heart Rate), lifestyle factors (Physical Activity Level, Quality of Sleep, Stress Level, Age, BMI Category, Gender), and a handful of occupations with genuine statistical signal (Nurse, Salesperson, Engineer, Teacher, Sales Representative, Lawyer).
- Held-out test performance: **92% accuracy**, macro F1 **0.90** — strong and balanced across all three classes, not just the majority "none" class.
- 5-fold cross-validated macro F1: **0.86 ± 0.05** — a more honest estimate than the single test split, confirming the result generalizes reasonably well given the dataset's size (374 rows).
- Confusion is small and mostly between **Insomnia and Sleep Apnea**, which is physiologically sensible since both share overlapping signals (heart rate, blood pressure) — the model rarely confuses either disorder with "no disorder."
- `Occupation_Doctor` scored well individually (chi²) but was dropped by RFECV, showing its signal was redundant once combined with other features (e.g. Age) — a good illustration of why combining filter, model-based, and wrapper methods gives a more reliable answer than any single one.